# Lab6.2: Topic modeling using gensim

In this notebook, we demonstrate how LDA models can be built and applied using the *gensim* package.

Credits:

This notebook is an adaptation of a blog from Susan Li's:

https://towardsdatascience.com/topic-modeling-and-latent-dirichlet-allocation-in-python-9bf156893c24



The data set we’ll use is a list of over one million news headlines published over a period of 15 years and can be downloaded from:

https://www.kaggle.com/therohk/million-headlines/data

You can also find this file in the lab6 folder.

We read the CSV file using the pandas framework.

In [1]:
import pandas as pd

#### Adapt the path below to point to your local copy of the data set
data = pd.read_csv('abcnews-date-text.csv', on_bad_lines="warn");
data_text = data[['headline_text']]
data_text['index'] = data_text.index
documents = data_text

Let's have a look at the data:

In [2]:
print(len(documents))
print(documents[:5])

1244184
                                       headline_text  index
0  aba decides against community broadcasting lic...      0
1     act fire witnesses must be aware of defamation      1
2     a g calls for infrastructure protection summit      2
3           air nz staff in aust strike for pay rise      3
4      air nz strike to affect australian travellers      4


We are going to use the *gensim* package to build our LDA models from the data.
Before building the model, we are going to preprocess the texts.

## Data Pre-processing
We will perform the following steps:

* Tokenization: Split the text into sentences and the sentences into words. Lowercase the words and remove punctuation.
* Words that have fewer than 3 characters are removed.
* All stopwords are removed.
* Words are lemmatized — words in third person are changed to first person and verbs in past and future tenses are changed into present.
* Words are stemmed — words are reduced to their root form.

In order to apply these processing steps, we first load the gensim and nltk libraries

In [3]:
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(2018)
import nltk
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sandy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
def lemmatize_stemming(text):
    return lemmatizer.lemmatize(text)
def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
           # result.append(token)
            result.append(lemmatize_stemming(token))
    return result

In [5]:
doc_sample = documents[documents['index'] == 4310].values[0][0]
print('original document: ')
words = []
for word in doc_sample.split(' '):
    words.append(word)
print(words)
print('\n\n tokenized and lemmatized document: ')
print(preprocess(doc_sample))

original document: 
['ratepayers', 'group', 'wants', 'compulsory', 'local', 'govt', 'voting']


 tokenized and lemmatized document: 
['ratepayer', 'group', 'want', 'compulsory', 'local', 'govt', 'voting']


We now apply the preprocessing to all the headlines and print the first 10 results

In [6]:
processed_docs = documents['headline_text'].map(preprocess)
### print the first 10 results
processed_docs[:10]

0          [decides, community, broadcasting, licence]
1                         [witness, aware, defamation]
2           [call, infrastructure, protection, summit]
3                          [staff, aust, strike, rise]
4              [strike, affect, australian, traveller]
5               [ambitious, olsson, win, triple, jump]
6          [antic, delighted, record, breaking, barca]
7    [aussie, qualifier, stosur, waste, memphis, ma...
8             [aust, address, security, council, iraq]
9                       [australia, locked, timetable]
Name: headline_text, dtype: object

## Bag of Words on the Data set
Create a dictionary from ‘processed_docs’ containing the number of times a word appears in the training set.
We are going to use the *Dictionary* function to derive a dictionary with counts from the headlines.

In [7]:
dictionary = gensim.corpora.Dictionary(processed_docs)
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 broadcasting
1 community
2 decides
3 licence
4 aware
5 defamation
6 witness
7 call
8 infrastructure
9 protection
10 summit


## Gensim filter_extremes
Filter out tokens that appear in
less than 15 documents (absolute number) or
more than 0.5 documents (fraction of total corpus size, not absolute number).
after the above two steps, keep only the first 100000 most frequent tokens.

In [8]:
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)

## Gensim doc2bow
For each document we create a dictionary reporting how many words and how many times those words appear. 
Gensim provides the *doc2bow* function to create a BoW vector representation for a document.
Save this to ‘bow_corpus’, then check our selected document earlier.

In [9]:
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
bow_corpus[4310]

[(164, 1), (241, 1), (615, 1), (891, 1), (4175, 1), (4176, 1), (4177, 1)]

Preview Bag Of Words for our sample preprocessed document.

In [10]:
bow_doc_4310 = bow_corpus[4310]
for i in range(len(bow_doc_4310)):
    print("Word {} (\"{}\") appears {} time.".format(bow_doc_4310[i][0], 
                                               dictionary[bow_doc_4310[i][0]], 
bow_doc_4310[i][1]))

Word 164 ("govt") appears 1 time.
Word 241 ("group") appears 1 time.
Word 615 ("local") appears 1 time.
Word 891 ("want") appears 1 time.
Word 4175 ("compulsory") appears 1 time.
Word 4176 ("ratepayer") appears 1 time.
Word 4177 ("voting") appears 1 time.


## TF-IDF
Create tf-idf model object using models.TfidfModel on ‘bow_corpus’ and save it to ‘tfidf’, then apply transformation to the entire corpus and call it ‘corpus_tfidf’. Finally we preview TF-IDF scores for our first document.


In [11]:
from gensim import corpora, models
tfidf = models.TfidfModel(bow_corpus)
corpus_tfidf = tfidf[bow_corpus]
from pprint import pprint
for doc in corpus_tfidf:
    pprint(doc)
    break

[(0, 0.6161125947380649),
 (1, 0.3308772069039591),
 (2, 0.5681053683635203),
 (3, 0.43379930266554434)]


## Running LDA using Bag of Words
Train our lda model using gensim.models.LdaMulticore and save it to ‘lda_model’. This takes a while.
Look at the documentation of *gensim* for further details:

https://radimrehurek.com/gensim/models/ldamulticore.html

As parameters, we pass the corpus data as BoW (a list of lists of tuples), the prefixed number of topics, the actual words and the number of passes and workers used for modeling.

In [12]:
lda_model = gensim.models.LdaMulticore(bow_corpus, num_topics=10, id2word=dictionary, passes=2, workers=2)

For each topic, we will explore the words occuring in that topic and its relative weight.

In [13]:
for idx, topic in lda_model.print_topics(-1):
    print('Topic: {} \nWords: {}'.format(idx, topic))

Topic: 0 
Words: 0.034*"case" + 0.025*"court" + 0.023*"police" + 0.021*"woman" + 0.019*"child" + 0.018*"vaccine" + 0.018*"murder" + 0.014*"death" + 0.014*"charged" + 0.014*"face"
Topic: 1 
Words: 0.037*"police" + 0.024*"school" + 0.015*"family" + 0.014*"missing" + 0.013*"guilty" + 0.011*"drum" + 0.010*"help" + 0.010*"farmer" + 0.010*"announces" + 0.010*"search"
Topic: 2 
Words: 0.019*"victorian" + 0.014*"premier" + 0.013*"claim" + 0.013*"time" + 0.012*"speaks" + 0.012*"hotel" + 0.012*"pandemic" + 0.011*"road" + 0.011*"say" + 0.009*"black"
Topic: 3 
Words: 0.023*"government" + 0.020*"coast" + 0.017*"national" + 0.016*"live" + 0.015*"plan" + 0.015*"federal" + 0.012*"gold" + 0.012*"care" + 0.010*"aged" + 0.010*"industry"
Topic: 4 
Words: 0.044*"covid" + 0.037*"coronavirus" + 0.019*"open" + 0.017*"lockdown" + 0.016*"melbourne" + 0.015*"dy" + 0.014*"final" + 0.014*"sydney" + 0.012*"hospital" + 0.012*"crash"
Topic: 5 
Words: 0.029*"election" + 0.023*"health" + 0.018*"say" + 0.018*"people" + 

Can you distinguish different topics using the words in each topic and their corresponding weights?

## Running LDA using TF-IDF

In [14]:
lda_model_tfidf = gensim.models.LdaMulticore(corpus_tfidf, num_topics=10, id2word=dictionary, passes=2, workers=4)
for idx, topic in lda_model_tfidf.print_topics(-1):
    print('Topic: {} Word: {}'.format(idx, topic))

Topic: 0 Word: 0.012*"government" + 0.008*"health" + 0.007*"federal" + 0.006*"wednesday" + 0.005*"mental" + 0.005*"funding" + 0.005*"weather" + 0.005*"extended" + 0.005*"christmas" + 0.005*"coronavirus"
Topic: 1 Word: 0.011*"climate" + 0.010*"people" + 0.008*"peter" + 0.007*"change" + 0.007*"violence" + 0.007*"domestic" + 0.006*"briefing" + 0.006*"november" + 0.006*"facebook" + 0.006*"james"
Topic: 2 Word: 0.030*"covid" + 0.025*"coronavirus" + 0.014*"donald" + 0.012*"country" + 0.011*"live" + 0.010*"restriction" + 0.010*"vaccine" + 0.009*"case" + 0.009*"hour" + 0.007*"victoria"
Topic: 3 Word: 0.011*"border" + 0.008*"pandemic" + 0.007*"rain" + 0.007*"queensland" + 0.007*"northern" + 0.007*"john" + 0.007*"bushfire" + 0.006*"coronavirus" + 0.005*"territory" + 0.005*"australia"
Topic: 4 Word: 0.024*"news" + 0.013*"drum" + 0.013*"rural" + 0.013*"interview" + 0.012*"scott" + 0.010*"andrew" + 0.009*"care" + 0.009*"tuesday" + 0.008*"aged" + 0.008*"thursday"
Topic: 5 Word: 0.011*"lockdown" + 0.

Again, can you distinguish different topics using the words in each topic and their corresponding weights? Do you observe any differences with the BoW version? Do these differences make sense given the information value weighing by the *tfidf* method?

## Performance evaluation by classifying sample document using LDA Bag of Words model
We will check where our test document would be classified.

In [15]:
processed_docs[4310]

['ratepayer', 'group', 'want', 'compulsory', 'local', 'govt', 'voting']

Document 4310 is already represented in the correct way. We can directly pass it to our *lda_model* to get the similarity scores for each topic. We represent each topic by printing 

In [16]:
for index, score in sorted(lda_model[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model.print_topic(index, 10)))


Score: 0.4009738862514496	 
Topic: 0.023*"government" + 0.020*"coast" + 0.017*"national" + 0.016*"live" + 0.015*"plan" + 0.015*"federal" + 0.012*"gold" + 0.012*"care" + 0.010*"aged" + 0.010*"industry"

Score: 0.2624129354953766	 
Topic: 0.037*"trump" + 0.035*"queensland" + 0.030*"victoria" + 0.025*"coronavirus" + 0.021*"covid" + 0.021*"australia" + 0.020*"news" + 0.019*"record" + 0.016*"market" + 0.014*"south"

Score: 0.24905632436275482	 
Topic: 0.029*"election" + 0.023*"health" + 0.018*"say" + 0.018*"people" + 0.017*"minister" + 0.016*"change" + 0.014*"morrison" + 0.012*"labor" + 0.012*"andrew" + 0.011*"country"

Score: 0.012508466839790344	 
Topic: 0.029*"home" + 0.021*"tasmania" + 0.018*"business" + 0.015*"royal" + 0.014*"return" + 0.012*"commission" + 0.012*"power" + 0.012*"fight" + 0.010*"town" + 0.009*"perth"

Score: 0.012508438900113106	 
Topic: 0.019*"border" + 0.017*"north" + 0.014*"china" + 0.014*"protest" + 0.012*"west" + 0.011*"amid" + 0.011*"say" + 0.011*"president" + 0.

Our test document has the highest probability to be part of the topic that our model assigned, which is the accurate classification.

### Analyzing our LDA model

Now that we have a trained model let’s visualize the topics for interpretability. 
To do so, we’ll use a popular visualization package, *pyLDAvis* which is designed to help interactively with:

1. Better understanding and interpreting individual topics, and
2. Better understanding the relationships between the topics.

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.
For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

You need to install *pyldavis* through the command line, following the instructions:

https://anaconda.org/conda-forge/pyldavis

WARNING: running the next cell takes a long time and you need some memory to run it. However, the result is spectacular.

In [19]:
%matplotlib inline
import pyLDAvis
import pyLDAvis.gensim_models
vis = pyLDAvis.gensim_models.prepare(topic_model=lda_model, corpus=bow_corpus, dictionary=dictionary)
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)

## Some other useful functions

In [20]:
#get the top 20 words and their weights for a specific topic
topic_id=1
top_terms=20
for wordid, score in lda_model.get_topic_terms(topic_id, top_terms):
    print(wordid, ":", dictionary[wordid], ":", score)

237 : police : 0.03691962
1116 : school : 0.023783095
805 : family : 0.014726741
313 : missing : 0.013643109
829 : guilty : 0.01252464
3276 : drum : 0.011305766
111 : help : 0.010277459
455 : farmer : 0.010272412
1023 : announces : 0.010091637
541 : search : 0.0100547355
361 : northern : 0.009511228
649 : dead : 0.009333553
746 : emergency : 0.009201409
195 : crash : 0.009181316
545 : finance : 0.009045832
284 : house : 0.008910879
9637 : daniel : 0.008909052
286 : white : 0.008081295
2054 : budget : 0.0076563717
1072 : fatal : 0.0074847457


In [21]:
#### Utility function to get the id for a word

def get_id_for_word(dictionary, word):
    for k, v in dictionary.iteritems():
        if (v==word):
            return k
    return -1

In [22]:
top_terms=20
index=get_id_for_word(dictionary,'market')
for topic_id, score in lda_model.get_term_topics(index):
    print("Topic:", topic_id)
    for wordid, score in lda_model.get_topic_terms(topic_id, top_terms):
        print(wordid, ":", dictionary[wordid], ":", score)


Topic: 6
8173 : trump : 0.0368245
2232 : queensland : 0.03498488
1619 : victoria : 0.030077033
18895 : coronavirus : 0.02463293
18896 : covid : 0.02124898
37 : australia : 0.021158813
1363 : news : 0.019586787
26 : record : 0.019303836
1188 : market : 0.015882794
144 : south : 0.014035122
679 : brisbane : 0.011738313
16 : australian : 0.009849701
3206 : street : 0.009503376
852 : price : 0.009491114
1422 : interview : 0.009080579
856 : india : 0.0076893554
1416 : warning : 0.0073620374
2264 : wall : 0.007202872
1415 : travel : 0.0071858396
3056 : video : 0.00700736


## Saving and loading your model for re-use

Building a model takes time.Once you have a stable model, you can save it to disk and reload it later.

In [23]:
# Save model to disk.
temp_file = "./model"
lda_model.save(temp_file)

# Load a potentially pretrained model from disk.
loaded_lda = LdaModel.load(temp_file)

c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\Sandy\anaconda3\envs\textmining\Lib\ast.py:587: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return Constant(*args, **kwargs)
c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
c:\Users\Sandy\anaconda3\envs\textmining\Lib\ast.py:587: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value in

NameError: name 'LdaModel' is not defined

## Testing model on unseen document

In [26]:
unseen_document = 'How a Pentagon deal became an identity crisis for Google'

In order to compare any new text against the topic model, we first need to process it in the same way as we processed the input texts for the model.
We apply the same preprocessing function and next apply the *doc2bow* function to represent it using the same vector representation as we used for modeling.

In [27]:
bow_vector = dictionary.doc2bow(preprocess(unseen_document))
print(bow_vector)


[(888, 1), (1088, 1), (1922, 1), (5668, 1), (12411, 1)]


We can now pass this representation of the unseen document into the model to compare it against all the topics.
The next function returns in index to the topics and a similarity score for the new document. We print the scores and the topics with the top 5 words.

In [28]:
for index, score in sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1]):
    print("Score: {}\t Topic_id {}\t Topic: {}".format(score, index, lda_model.print_topic(index, 5)))

Score: 0.34986016154289246	 Topic_id 8	 Topic: 0.055*"australia" + 0.025*"donald" + 0.018*"restriction" + 0.013*"australian" + 0.013*"world"
Score: 0.3139866590499878	 Topic_id 6	 Topic: 0.037*"trump" + 0.035*"queensland" + 0.030*"victoria" + 0.025*"coronavirus" + 0.021*"covid"
Score: 0.2194429486989975	 Topic_id 9	 Topic: 0.019*"border" + 0.017*"north" + 0.014*"china" + 0.014*"protest" + 0.012*"west"
Score: 0.01667405106127262	 Topic_id 5	 Topic: 0.029*"election" + 0.023*"health" + 0.018*"say" + 0.018*"people" + 0.017*"minister"
Score: 0.016673879697918892	 Topic_id 3	 Topic: 0.023*"government" + 0.020*"coast" + 0.017*"national" + 0.016*"live" + 0.015*"plan"
Score: 0.01667313650250435	 Topic_id 0	 Topic: 0.034*"case" + 0.025*"court" + 0.023*"police" + 0.021*"woman" + 0.019*"child"
Score: 0.016672290861606598	 Topic_id 1	 Topic: 0.037*"police" + 0.024*"school" + 0.015*"family" + 0.014*"missing" + 0.013*"guilty"
Score: 0.016672290861606598	 Topic_id 2	 Topic: 0.019*"victorian" + 0.014*"

This text matches best with topic 5 although the score is not very high!

### Updating the model with a new document

We can also use the unseen documents to extend our model and update the topics. This is useful when processing texts in a stream.

In [29]:
# Update the model by incrementally training on the new corpus.

other_texts = [['computer', 'time', 'graph'],['survey', 'response', 'eps'],['human', 'system', 'computer']]
other_corpus = [dictionary.doc2bow(text) for text in other_texts]

# Update the model by incrementally training on the new corpus.
lda_model.update(other_corpus)  # update the LDA model with additional documents



c:\Users\Sandy\anaconda3\envs\textmining\Lib\site-packages\gensim\models\ldamodel.py:850: RuntimeWarning: overflow encountered in exp2
  perwordbound, np.exp2(-perwordbound), len(chunk), corpus_words


## End of this notebook